# ⚙️ Prerequisites — Seed Production Traces
## Microsoft Build 2026 | LAB231 — Phase 0

---

### Why this step?

Before we can evaluate or fine-tune anything, we need the **production agent to generate traces**.
This notebook runs the production agent (GPT-5.4) against our seed dataset — generating the
correct tool-call sequences that become:

1. **Training data** for SFT distillation (Phase 2)
2. **Baseline evaluation** to see how the prod agent performs

### What this notebook does:

| # | Step | What happens |
|---|------|-------------|
| 1 | Setup | Connect to Azure AI Foundry project |
| 2 | Upload seed data | Push seed scenarios to the project |
| 3 | Create evaluation | Define grader + criteria |
| 4 | Run prod agent | Generate traces against all seed scenarios |
| 5 | Monitor & results | Wait for completion, view baseline scores |

### Prerequisites:

- ✅ `.env` file with `PROJECT_ENDPOINT` configured
- ✅ Agents deployed via `cd deploy && azd up`
- ✅ Function app running at [retail-tools-omkarm.azurewebsites.net](https://retail-tools-omkarm.azurewebsites.net/demo)

> 💡 **This is a one-time setup step.** Once traces are generated, you can skip to `1_introduction.ipynb`.

---

## 1. Setup & Configuration

In [1]:
import os
import json
import requests
import time
from datetime import datetime
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
if not PROJECT_ENDPOINT:
    raise ValueError("PROJECT_ENDPOINT not set. Copy .env.example → .env and fill in your endpoint.")

parts = PROJECT_ENDPOINT.split("/")
ACCOUNT = parts[2].split(".")[0]
PROJECT = parts[-1]
BASE_URL = f"{PROJECT_ENDPOINT}/openai"

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
DATASET_NAME = f"retail-seed-{TIMESTAMP}"
EVAL_NAME = f"retail-seed-eval-{TIMESTAMP}"

credential = DefaultAzureCredential()

def get_headers():
    token = credential.get_token("https://ai.azure.com/.default").token
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"✅ Connected to: {ACCOUNT}/{PROJECT}")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Eval: {EVAL_NAME}")

✅ Connected to: ai-account-44mf5lkxqssxm/ai-project-omi-build26-azd-env
   Dataset: retail-seed-20260530-091338
   Eval: retail-seed-eval-20260530-091338


---

## 2. Upload Seed Dataset

The seed dataset ([`data/retail_seed.jsonl`](../data/retail_seed.jsonl)) contains scenarios
that the production agent will process. Each scenario includes:
- Customer message (the input)
- Expected resolution, actions, and amounts (for grading)

In [2]:
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

dataset = client.datasets.upload_file(
    file_path="../data/retail_seed.jsonl",
    name=DATASET_NAME,
    version="1.0"
)

DATASET_URI = f"azureai://accounts/{ACCOUNT}/projects/{PROJECT}/data/{dataset.name}/versions/{dataset.version}"
print(f"✅ Dataset uploaded: {dataset.name}")
print(f"   URI: {DATASET_URI}")

✅ Dataset uploaded: retail-seed-20260530-091338
   URI: azureai://accounts/ai-account-44mf5lkxqssxm/projects/ai-project-omi-build26-azd-env/data/retail-seed-20260530-091338/versions/1.0


---

## 3. Create Evaluation with Grader

We set up 3 evaluation criteria:
- **retail_quality** — custom Python grader ([`scripts/retail_grader_response.py`](../scripts/retail_grader_response.py))
- **IntentResolution** — built-in Azure AI evaluator
- **TaskCompletion** — built-in Azure AI evaluator

In [3]:
# Load the custom grader
with open("../scripts/retail_grader_response.py", "r") as f:
    GRADER_SOURCE = f.read()

print(f"✅ Loaded grader ({len(GRADER_SOURCE)} chars)")

# Create evaluation definition
eval_body = {
    "name": EVAL_NAME,
    "data_source_config": {
        "type": "custom",
        "include_sample_schema": True,
        "item_schema": {
            "type": "object",
            "properties": {
                "messages": {"type": "array"},
                "expected_resolution": {"type": "string"},
                "expected_actions": {"type": "object"},
                "expected_amounts": {"type": "object"}
            }
        }
    },
    "testing_criteria": [
        {
            "name": "retail_quality",
            "type": "python",
            "source": GRADER_SOURCE,
            "pass_threshold": 0.8
        },
        {
            "type": "azure_ai_evaluator",
            "name": "IntentResolution",
            "evaluator_name": "builtin.intent_resolution",
            "initialization_parameters": {"deployment_name": "gpt-5.4"},
            "data_mapping": {
                "query": "{{item.messages}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.expected_resolution}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "name": "TaskCompletion",
            "evaluator_name": "builtin.task_completion",
            "initialization_parameters": {"deployment_name": "gpt-5.4"},
            "data_mapping": {
                "query": "{{item.messages}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.expected_resolution}}"
            }
        }
    ]
}

url = f"{BASE_URL}/evals?api-version=2025-11-15-preview"
resp = requests.post(url, headers=get_headers(), json=eval_body)
if resp.status_code in [200, 201]:
    eval_id = resp.json()["id"]
    print(f"✅ Evaluation created: {eval_id}")
    print(f"   Criteria: retail_quality + IntentResolution + TaskCompletion")
else:
    print(f"❌ Failed: {resp.status_code}")
    print(resp.text[:300])

✅ Loaded grader (20778 chars)
✅ Evaluation created: eval_edbedaccbcca427b83a0b8e38c17aadd
   Criteria: retail_quality + IntentResolution + TaskCompletion


---

## 4. Run Production Agent Against Seed Data

We run the production agent (`retail-prod` / GPT-5.4) against all seed scenarios.
This generates the traces that become training data for SFT.

> ⏱️ This takes **15-30 minutes** depending on scenario count and TPM.

In [4]:
# Run the production agent to generate seed traces
AGENTS = {
    "prod": ("retail", None),
}

run_ids = {}
for model_name, (agent_name, version) in AGENTS.items():
    target = {"type": "azure_ai_agent", "name": agent_name}
    if version:
        target["version"] = version

    run_body = {
        "name": f"seed-{model_name}-{TIMESTAMP}",
        "data_source": {
            "type": "azure_ai_target_completions",
            "source": {"type": "file_id", "id": DATASET_URI},
            "target": target,
            "input_messages": {"type": "item_reference", "item_reference": "item.messages"}
        }
    }
    url = f"{BASE_URL}/evals/{eval_id}/runs?api-version=2025-11-15-preview"
    resp = requests.post(url, headers=get_headers(), json=run_body)
    if resp.status_code in [200, 201]:
        run_ids[model_name] = resp.json()["id"]
        print(f"  🚀 {model_name} → {run_ids[model_name]}")
    else:
        print(f"  ❌ {model_name}: {resp.status_code} - {resp.text[:150]}")

print(f"\n✅ Seed run launched")
print(f"⏱️  Monitor progress in Azure AI Foundry → Evaluations")

  🚀 prod → evalrun_c8c7e9b36e1a4e2ea1704ca82dc24178

✅ Seed run launched
⏱️  Monitor progress in Azure AI Foundry → Evaluations


---

## 5. Monitor Progress

Wait for the seed run to complete. You can also monitor in the
[Azure AI Foundry → Evaluations](https://ai.azure.com) tab.

In [ ]:
# Poll until complete
print("Waiting for seed run to complete...")
start_time = time.time()

while True:
    done = 0
    for model_name, run_id in run_ids.items():
        url = f"{BASE_URL}/evals/{eval_id}/runs/{run_id}?api-version=2025-11-15-preview"
        resp = requests.get(url, headers=get_headers())
        status = resp.json().get("status", "")
        if status in ("completed", "succeeded", "failed"):
            done += 1
    elapsed = int(time.time() - start_time)
    print(f"  [{elapsed//60:02d}:{elapsed%60:02d}] {done}/{len(run_ids)} complete", end="\r")
    if done == len(run_ids):
        break
    time.sleep(30)

print(f"\n✅ Seed run finished in {elapsed//60}m {elapsed%60}s")

---

## 6. View Baseline Results

In [ ]:
# Fetch results
for model_name, run_id in run_ids.items():
    url = f"{BASE_URL}/evals/{eval_id}/runs/{run_id}?api-version=2025-11-15-preview"
    resp = requests.get(url, headers=get_headers())
    data = resp.json()
    
    print("\n" + "=" * 60)
    print(f"  PRODUCTION AGENT BASELINE — {model_name}")
    print("=" * 60)
    print(f"  Status: {data.get('status')}")
    
    rc = data.get("result_counts", {})
    print(f"  Total: {rc.get('total', 0)} | Passed: {rc.get('passed', 0)} | Failed: {rc.get('failed', 0)}")
    
    print(f"\n  Per-criteria breakdown:")
    for criteria in data.get("per_testing_criteria_results", []):
        cname = criteria.get("testing_criteria", "")
        passed = criteria.get("passed", 0)
        failed = criteria.get("failed", 0)
        total = passed + failed
        score = passed / total * 100 if total > 0 else 0
        print(f"    {cname:<20s} {passed}/{total} ({score:.0f}%)")
    
    report_url = data.get("report_url", "")
    if report_url:
        print(f"\n  🔗 Report: {report_url}")


---

## ✅ Done! Traces Generated

The production agent has processed all seed scenarios. The traces are now available in Azure AI Foundry.

### What's next?

| Notebook | What it does |
|----------|-------------|
| [`1_introduction.ipynb`](./1_introduction.ipynb) | Run base model evaluations and see the leaderboard |
| [`2_sft_distillation.ipynb`](./2_sft_distillation.ipynb) | Use these traces to fine-tune smaller models |

> 💡 The traces from this run serve as both **evaluation baseline** (how good is the prod agent?)  
> and **training data** (teach smaller models to replicate this behavior).